<div style="width: 30em; float: right; padding: 3em; border: 5px red solid; background-color: darkred; color: white"><p style="font-size: large; font-weight:bold">Rename this notebook before running any cells!</p><ol><li>Right-click on the tab title above or on the notebook in the file-browser on the left and select rename.</li><li>Remove the "_orig" part of the file name.</li></ol><p style="font-size: large; font-weight:bold">Check the notebook kernel!</p><ol><li>Check the current notebook kernel in the upper right corner.</li><li>Set to "Python 3.12 (Conda)".</li></ol></div>

# Session 3 — Loops and Core Data Structures

IFI 8410 · Module 1

Everything in this notebook answers three questions about one small dataset:

> **What sells most often, when do sales peak, and which customer groups buy which items?**

The dataset is a week of transactions at a campus coffee cart. It is small
enough to read with your own eyes, which means you can always check whether the
program's answer is right — and that is exactly what you want while you are
learning the tools that produce the answer.

Along the way the notebook covers the four containers Python gives you, the two
kinds of loop, and four patterns that reappear in nearly every program you will
write this term.

By the end you should be able to:

- choose between a **list**, **tuple**, **set**, and **dictionary**, and say why
- write a `for` loop over a collection and a `while` loop over a condition
- apply the four core loop patterns — **counting**, **accumulating**,
  **filtering**, and **finding a maximum**
- build a counts dictionary by hand, then with `.get()`, then with `Counter`
- explain what a library function like `value_counts()` is doing for you later
  in the term

**How to work through it.** Run every code cell in order, top to bottom. Cells
later in the notebook use variables defined earlier, so skipping around will
produce `NameError`. Where you see a **Try it** cell, actually try it — the
prediction you make before running is worth more than the output.

---

## 1. Refresher: values, types, expressions

Before loops, a quick check that Sessions 1–2 are in place. The cell below uses
every fundamental this session builds on:

| Element | In the cell below | What it is |
|---|---|---|
| **Variable** | `student_name` | a name bound to a value |
| **String** | `"Amina"` | text, in quotes |
| **Integer** | `3` | a whole number |
| **Float** | `3.50` | a number with a fractional part |
| **Boolean** | `True` | one of exactly two truth values |
| **Expression** | `items_bought * coffee_price` | code that produces a value |
| **Conditional** | `if ... else ...` | chooses which statement runs |
| **f-string** | `f"{name} owes ${total:.2f}"` | a string with values interpolated |

Two details in the f-string are worth naming. `{total_due:.2f}` is a **format
specification**: `.2f` means "as a fixed-point number with two digits after the
decimal point". It rounds for display only — it does not change the value stored
in `total_due`. And the `$` immediately before the `{` is just a literal dollar
sign; it has no special meaning to Python.

In [ ]:
# Session 1–2 refresher:
# variables, strings, numbers, Booleans, f-strings, arithmetic, conditionals

student_name = "Amina"
items_bought = 3
coffee_price = 3.50
has_student_discount = True

subtotal = items_bought * coffee_price

if has_student_discount:
    total_due = subtotal * 0.90
else:
    total_due = subtotal

print(f"{student_name} owes ${total_due:.2f}")
print(type(student_name))
print(type(items_bought))
print(type(total_due))
print(type(has_student_discount))

`type()` reports the type of a value. Notice that `subtotal` came out as a
`float` even though `items_bought` was an `int`: multiplying an `int` by a
`float` produces a `float`. Python decides types from the values, not from a
declaration you write.

**Teaching prompt.** What changes if `items_bought` is the string `"3"` rather
than the integer `3`?

The multiplication does not fail — but it does something you almost certainly
did not want. `"3" * 3.50` is an error, while `"3" * 3` gives `"333"`, because
`*` on a string means *repeat*. This is the single most common source of
confusing bugs when a value arrives from user input or a file, where everything
starts out as text.

**Second prompt.** What does the `if` statement control? Not the value of
`subtotal` — that is already computed. It controls *which one* of the two
assignment statements to `total_due` runs. Exactly one of them does.

In [ ]:
### Try it: change `items_bought` to the string "3" and re-run.
### Then change `has_student_discount` to False. Predict each result first.

### Enter your code here ###

---

## 2. The dataset: coffee-cart transactions

Here is the data the rest of the notebook analyzes. Read the structure before
you read the code that uses it.

Each transaction is a **dictionary**, because a transaction has *named fields* —
`item`, `price`, `day`, `hour`, and so on. Names are the right tool here: field
number 4 means nothing to a reader, but `sale["hour"]` means something to
everyone.

The transactions are collected in a **list**, because the collection is ordered,
can contain duplicates (two identical coffee purchases are two real sales), and
could grow as the day goes on.

In [ ]:
# A small, deliberately human-readable dataset.
# Each dictionary represents one transaction at a campus coffee cart.

sales = [
    {"id": 101, "item": "coffee", "category": "drink", "price": 3.50, "day": "Mon", "hour": 8,  "customer_type": "student"},
    {"id": 102, "item": "tea",    "category": "drink", "price": 2.75, "day": "Mon", "hour": 9,  "customer_type": "faculty"},
    {"id": 103, "item": "muffin", "category": "food",  "price": 2.50, "day": "Mon", "hour": 9,  "customer_type": "student"},
    {"id": 104, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "hour": 8,  "customer_type": "student"},
    {"id": 105, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Tue", "hour": 10, "customer_type": "staff"},
    {"id": 106, "item": "coffee", "category": "drink", "price": 3.50, "day": "Tue", "hour": 10, "customer_type": "faculty"},
    {"id": 107, "item": "tea",    "category": "drink", "price": 2.75, "day": "Wed", "hour": 8,  "customer_type": "student"},
    {"id": 108, "item": "cookie", "category": "food",  "price": 1.75, "day": "Wed", "hour": 11, "customer_type": "student"},
    {"id": 109, "item": "coffee", "category": "drink", "price": 3.50, "day": "Wed", "hour": 11, "customer_type": "staff"},
    {"id": 110, "item": "muffin", "category": "food",  "price": 2.50, "day": "Thu", "hour": 9,  "customer_type": "faculty"},
    {"id": 111, "item": "coffee", "category": "drink", "price": 3.50, "day": "Thu", "hour": 9,  "customer_type": "student"},
    {"id": 112, "item": "bagel",  "category": "food",  "price": 3.00, "day": "Fri", "hour": 8,  "customer_type": "student"},
]

print(f"Number of transactions: {len(sales)}")
print("First transaction:")
print(sales[0])

The structure is a **list of dictionaries**, and it is worth being precise about
what each layer does:

- The **outer list** preserves transaction order and allows duplicates.
- Each **inner dictionary** connects field names to field values.
- Together they resemble a tiny table: each dictionary is one **row**, and each
  key is a **column name**.

That last point matters more than it looks. In Session 8 you will load a real
CSV file into a pandas `DataFrame`, and a `DataFrame` is — conceptually — this
same list of rows with named columns. Everything you write by hand here has a
one-line equivalent there. Writing it by hand first is what makes the one-liner
comprehensible rather than magical.

`sales[0]` reads position 0 of the list — the first transaction. `len(sales)`
reports how many items the list holds. Both work on any list.

In [ ]:
# Reaching into the structure: list position first, then dictionary key.

first_sale = sales[0]          # a dictionary
print(first_sale)

print(first_sale["item"])      # a value inside that dictionary
print(sales[0]["item"])        # the same thing, in one step
print(sales[-1]["item"])       # the last transaction's item

`sales[0]["item"]` reads left to right: take position 0 of `sales`, which gives
a dictionary, then take the value stored under the key `"item"`. Each pair of
square brackets peels off one layer.

Note that the two uses of `[...]` look identical but mean different things:
`[0]` is a **position** in a list, `["item"]` is a **key** in a dictionary. Lists
are indexed by position; dictionaries are indexed by key.

---

# Core containers

Python's four built-in containers are deliberately different from one another.
Choosing the right one is a design decision that tells a future reader what you
believe about your data.

| Container | Written as | Ordered | Changeable | Duplicates | Use it when |
|---|---|---|---|---|---|
| **list** | `[1, 2, 3]` | yes | yes | yes | order matters, contents may change |
| **tuple** | `(1, 2, 3)` | yes | **no** | yes | a fixed record that should not change |
| **set** | `{1, 2, 3}` | no | yes | **no** | you need distinct values or fast membership tests |
| **dict** | `{"a": 1}` | yes (insertion) | yes | keys unique | you look values up by a name or id |

Reference: the [Python tutorial on data
structures](https://docs.python.org/3/tutorial/datastructures.html).

## 3. Lists: ordered, mutable sequences

Reach for a list when **order matters**, **duplicates are meaningful**, or the
collection **needs to change**.

Two rules about positions cause most beginner errors, so learn them now:

1. Indices start at **0**. The first item is `items[0]`, and the last item of a
   list of length *n* is `items[n - 1]`.
2. Slicing **excludes** the ending position. `items[1:4]` gives positions 1, 2,
   and 3 — three items, not four.

Negative indices count from the end: `items[-1]` is the last item, `items[-2]`
the one before it.

In [ ]:
# Extract item names into a list.
items_sold = []

for sale in sales:
    items_sold.append(sale["item"])

print(items_sold)
print(items_sold[0])       # First item
print(items_sold[-1])      # Last item
print(items_sold[1:4])     # Positions 1, 2, and 3; position 4 is excluded
print(len(items_sold))

That loop is the **accumulator pattern applied to a list**, and it has three
parts you will use constantly:

1. **Before the loop**, create an empty list: `items_sold = []`.
2. **Inside the loop**, `append` one value per pass.
3. **After the loop**, the list holds one entry per transaction.

`.append()` adds a single item to the end of the list and returns `None`. A
frequent mistake is writing `items_sold = items_sold.append(x)`, which throws
away the list and leaves you with `None`. Call it as a statement, not as an
expression.

Note that `items_sold` has 12 entries, the same as `sales`, and it contains
`"coffee"` several times. Duplicates are not a defect here — each one is a real
sale.

In [ ]:
# Mutating a list in place

menu_items = ["coffee", "tea", "muffin"]
print("Before:", menu_items)

menu_items.append("bagel")
menu_items[0] = "espresso"

print("After:", menu_items)

**Mutable** means the object itself can be changed after it is created. Both
operations above modify the *existing* list rather than producing a new one:
`.append()` makes it longer, and `menu_items[0] = "espresso"` replaces what is at
position 0.

This has a consequence worth knowing early. If two variables refer to the same
list, a change through one is visible through the other — there is only one
list. The next cell shows both behaviors side by side.

In [ ]:
# Two names for ONE list, versus a genuine copy.

original = ["coffee", "tea"]
same_list = original            # NOT a copy — another name for the same list
a_copy = original.copy()        # a genuine, independent copy

original.append("muffin")

print("original: ", original)
print("same_list:", same_list)   # changed too
print("a_copy:   ", a_copy)      # unchanged

In [ ]:
# Building a NEW list instead of changing the original list

prices = [sale["price"] for sale in sales]
discounted_prices = []

for price in prices:
    discounted_prices.append(round(price * 0.90, 2))

print("Original prices:  ", prices)
print("Discounted prices:", discounted_prices)

Two things are new in that cell.

`[sale["price"] for sale in sales]` is a **list comprehension** — a compact way
to write the build-a-list loop you saw above. Read it as "the value of
`sale["price"]`, for each `sale` in `sales`". It is exactly equivalent to a
three-line `for` loop with `.append()`, and you should be comfortable writing
both. Section 11 returns to comprehensions with a filter attached.

`round(value, 2)` rounds to two decimal places and **returns a new number**,
leaving the original untouched. It is needed here because a 10% discount on
`$2.75` is `2.475` — a price in fractions of a cent. Rounding brings the result
back to money.

The important habit: `prices` is untouched. Building a new list rather than
editing the old one keeps the original data available for later questions.

In [ ]:
# Positions, the last valid index, and the off-by-one error.

days = ["Mon", "Tue", "Wed", "Thu", "Fri"]

print(len(days))   # 5 items
print(days[0])     # first item  -> Mon
print(days[4])     # fifth item  -> Fri, the LAST valid index

# The line below raises IndexError because valid positions are 0 through 4.
# Uncomment it, run the cell, and read the error message carefully.
# print(days[5])

# range(5) produces 0, 1, 2, 3, 4 — exactly the valid positions.
for index in range(len(days)):
    print(index, days[index])

**Prompt.** Why is `days[5]` invalid even though `len(days)` is 5?

Because counting starts at 0. Five items occupy positions 0, 1, 2, 3, 4 — the
last index is always `len(days) - 1`. This mismatch between "how many" and "the
last position" is the classic **off-by-one error**, and `IndexError: list index
out of range` is Python telling you it happened.

Notice that `range(len(days))` produces precisely `0, 1, 2, 3, 4`, which is why
that idiom is safe. But when you only need the values and not the positions,
`for day in days:` is clearer — prefer it.

In [ ]:
### Try it: print the LAST item of `days` in two different ways,
### one using len() and one using a negative index.

### Enter your code here ###

---

## 4. Tuples: fixed records

A tuple is a compact record with **fixed positions**. A location, for instance,
might always consist of `(building, floor, room)`. Written with parentheses, a
tuple is indexed exactly like a list — but it cannot be changed after creation.

Why would anyone want a container that *cannot* be changed? Because
immutability is a statement of intent. Using a tuple tells the next reader — and
your future self — "these fields belong together, and this record is not
supposed to be edited". That guarantee also lets Python use tuples in places a
list cannot go, such as a dictionary key.

In [ ]:
# A tuple represents a fixed record.

coffee_cart_location = ("Technology Building", 1, "Lobby")

building = coffee_cart_location[0]
floor = coffee_cart_location[1]
area = coffee_cart_location[2]

print(f"The coffee cart is in the {building}, floor {floor}, near the {area}.")

In [ ]:
# Tuple unpacking makes fixed records readable.

item_summary = ("coffee", 4, 14.00)

item_name, number_sold, revenue = item_summary

print(f"{item_name}: {number_sold} sold, ${revenue:.2f} in revenue")

**Unpacking** assigns the three elements of the tuple to three variables in one
statement. The number of names on the left must match the number of elements on
the right, or Python raises `ValueError`.

Unpacking is not a curiosity — it is everywhere in Python. When you write
`for item, price in menu_prices.items():` later in this notebook, that is tuple
unpacking: `.items()` hands you a `(key, value)` tuple each time round the loop,
and the two names on the left pull it apart. Recognizing it here means that line
will not look like new syntax when you meet it.

In [ ]:
# Tuples cannot be modified in place.

days_open = ("Mon", "Tue", "Wed", "Thu", "Fri")
print(days_open)
print(len(days_open))
print(days_open[0])

# Uncomment to see the error Python raises:
# days_open[0] = "Monday"

The error is `TypeError: 'tuple' object does not support item assignment`. Note
what the message says — it is not that the operation is forbidden by a rule
somewhere, but that tuples simply have no such capability.

**Prompt.** Which is a better representation for the five-day operating
schedule: a list or a tuple?

A tuple. The cart's operating days are a fixed fact about the business, not a
collection that accumulates during the program. Choosing a tuple communicates
that, and it turns an accidental `days_open[0] = ...` into an immediate error
instead of silently corrupted data. Choose a list when you expect to add,
remove, or reorder; choose a tuple when the record is settled.

---

## 5. Sets: uniqueness and membership

A set holds **unique values only** and answers two kinds of question well:

- *Which distinct values occurred?*
- *Is this value present?*

A set has **no meaningful positional order**, so `my_set[0]` is an error — do
not write it. When a set prints, the order you see is an artifact of how Python
stores the values; do not rely on it, and sort the values when you need
predictable output.

In [ ]:
# Duplicate item names occur in the sales list.
# A set retains each distinct item only once.

unique_items = set(items_sold)

print("All items sold:", items_sold)
print("Distinct items:", unique_items)
print(f"Number of distinct items: {len(unique_items)}")

# For output people will read, sort it into a list.
print("Distinct items, sorted:", sorted(unique_items))

`set(items_sold)` builds a set **from** the list, silently dropping repeats: 12
sales, 5 distinct items. This is the shortest correct answer to "what did we
sell this week?"

`sorted(...)` takes any collection and returns a new **list** in sorted order.
Use it whenever printed output must be reproducible — a set's own display order
can differ between runs and machines, which makes results hard to compare and
hard to test.

In [ ]:
# Membership test with `in`

if "coffee" in unique_items:
    print("Coffee was sold this week.")

if "smoothie" not in unique_items:
    print("Smoothies were not sold this week.")

`in` and `not in` produce Booleans, which is why they fit directly into an `if`.
They work on lists too — `"coffee" in items_sold` is perfectly valid — but there
is a difference in how the work is done.

Checking a **list** means scanning it item by item until a match is found. With
12 items that is instant; with 12 million it is not. Checking a **set** uses
hashing to jump more or less straight to the answer, regardless of size. You do
not need the details now (Session 11 covers hashing), only the rule of thumb:
**if the program's main job is asking "is this value present?", store the values
in a set.**

In [ ]:
# Sets are useful for data-quality questions.

reported_categories = ["drink", "food", "drink", "food", "drink"]
unique_categories = set(reported_categories)

print(unique_categories)

# Do not rely on a set's display order.
# The important fact is uniqueness, not position.

This is a small but genuinely practical use. Turning a column into a set tells
you, at a glance, whether it contains what you expect. If `unique_categories`
came back as `{'drink', 'food', 'Food', 'drinks', ''}` you would have found a
data-entry problem in one line — inconsistent capitalization, an inconsistent
plural, and an empty value — before it silently corrupted every count you
computed afterwards.

---

## 6. Dictionaries: key-to-value lookup

Use a dictionary when you have a **unique identifier or label** and need the
matching value. Keys must be unique; assigning under an existing key **replaces**
the previous value rather than adding a second entry.

Think of it as a lookup table: the key is what you know, the value is what you
want.

In [ ]:
# A dictionary maps unique keys to values.

menu_prices = {
    "coffee": 3.50,
    "tea": 2.75,
    "muffin": 2.50,
    "bagel": 3.00,
    "cookie": 1.75,
}

print(menu_prices["coffee"])
print(menu_prices["muffin"])
print(len(menu_prices))

The syntax is `{key: value, key: value, ...}`. Here every key is a string and
every value is a float, but that is a choice, not a requirement — values can be
anything at all, including lists and other dictionaries. Keys must be
*immutable*, which is why a string or a tuple can be a key but a list cannot.

`menu_prices["coffee"]` is a **lookup**. If the key is missing, Python raises
`KeyError` — not `None`, and not zero. Section 6.3 shows the safe alternative.

In [ ]:
# Keys are unique: a second assignment replaces the old value.

print("Before:", menu_prices["coffee"])

menu_prices["coffee"] = 3.75

print("After: ", menu_prices["coffee"])
print(len(menu_prices), "items on the menu — still five, not six")

The same statement form does two different jobs depending on whether the key
already exists: if it does, the value is **replaced**; if it does not, a new
entry is **created**. That single behavior is what makes the counting idiom in
Section 13 work.

In [ ]:
# Safe lookup with `.get()`

requested_item = "smoothie"

price = menu_prices.get(requested_item)

if price is None:
    print(f"{requested_item.title()} is not on the menu.")
else:
    print(f"{requested_item.title()} costs ${price:.2f}")

`.get(key)` returns the value if the key is present and `None` if it is not —
where `menu_prices["smoothie"]` would crash with `KeyError`. Use `.get()`
whenever a missing key is a *normal* situation your program should handle rather
than a bug it should report.

`.get(key, default)` takes a second argument giving the value to return instead
of `None`. `menu_prices.get("smoothie", 0.0)` yields `0.0`. Section 15 turns that
into a one-line counting idiom.

Two smaller points: `is None` is the correct test for `None` (use `is`, not
`==`), and `.title()` is a string method returning a Title-Cased copy — the
original string is unchanged, because strings, like tuples, are immutable.

In [ ]:
# Iterating over keys and values with `.items()`

for item, price in menu_prices.items():
    print(f"{item.title():<10} ${price:.2f}")

`.items()` yields one `(key, value)` tuple per entry, and `for item, price in ...`
unpacks each tuple into two variables — the same unpacking you met with tuples.

There are three ways to loop over a dictionary, and picking the right one makes
your intent obvious:

| Loop | Gives you | Use when |
|---|---|---|
| `for k in d:` | keys | you only need the keys |
| `for v in d.values():` | values | you only need the values |
| `for k, v in d.items():` | both | you need the pairing |

In the f-string, `{item.title():<10}` combines an expression with a format
specification: `<10` means "left-align in a field 10 characters wide", which is
what lines the prices up into a column. `>10` would right-align, and `^10` would
center.

In [ ]:
### Try it: print every menu item that costs less than $3.00,
### using .items() and an if statement.

### Enter your code here ###

---

# Iteration: the two kinds of loop

## 7. `for` loops: a known collection, repeated work

A `for` loop is the natural choice when Python can iterate over a **known
collection** — the transactions in `sales`, the characters in a string, or the
values produced by `range()`. You are saying: *do this once for each item*.

The header `for sale in sales:` creates the variable `sale` and binds it to a
different element each time round. You do not manage a counter, and you cannot
run off the end.

In [ ]:
# Iterate over a list of dictionaries.

for sale in sales:
    print(f'{sale["day"]} at {sale["hour"]}:00 — {sale["item"]}')

Each pass gives one dictionary, and the body reads three of its fields.

One syntax detail: this f-string is delimited by **single** quotes so that the
**double** quotes around the dictionary keys sit inside it without conflict.
Mixing them the other way round works equally well; mixing them carelessly
produces a `SyntaxError`.

In [ ]:
# Iterate over characters in a string.

item = "coffee"

for letter in item:
    print(letter)

A string is iterable too — looping over it yields one character at a time. This
is worth seeing because it explains an error you will eventually hit: if a
variable you thought held a list of names actually holds one long string,
looping over it gives you letters, not names.

In [ ]:
# `range()` is useful for a known number of repetitions.

for number in range(1, 6):
    print(f"Transaction sample number {number}")

`range()` generates a sequence of integers and comes in three forms:

| Call | Produces |
|---|---|
| `range(5)` | 0, 1, 2, 3, 4 |
| `range(1, 6)` | 1, 2, 3, 4, 5 |
| `range(0, 10, 2)` | 0, 2, 4, 6, 8 |

The **stop value is always excluded** — the same convention as slicing. That is
why `range(1, 6)` is what you want when counting 1 through 5, and why `range(n)`
gives exactly the valid indices of a list of length `n`.

Use `range()` when you need repetitions or numbers. When you have a collection,
loop over the collection itself.

In [ ]:
# `enumerate()` provides both position and value.

for position, sale in enumerate(sales, start=1):
    print(f'{position}. {sale["item"]} on {sale["day"]}')

`enumerate()` hands you a `(position, value)` tuple each pass, which the two
names unpack. By default positions start at 0; `start=1` makes them start at 1,
which is what people expect in a numbered list on screen.

This replaces the clumsier `for i in range(len(sales)):` followed by
`sales[i]` — you get the counter *and* the item without the risk of an
off-by-one error.

---

## 8. `while` loops: unknown repetitions, controlled by a condition

A `while` loop runs **as long as its condition remains true**. Use it when you
do not know the repetition count in advance — reading records until a sentinel
value appears, or re-prompting until the input is valid.

Every correct `while` loop has three parts, and leaving out the third is the
classic beginner bug:

1. **Initialize** the state before the loop.
2. **Test** a condition that depends on that state.
3. **Update** the state inside the body, so the condition can eventually become
   false.

In [ ]:
# Simulate receiving transactions until the cart closes.
# The number of processed records is controlled by a condition.

index = 0                        # 1. initialize

while index < len(sales):        # 2. test
    
    sale = sales[index]
    print(f'Processing transaction {sale["id"]}: {sale["item"]}')
    index = index + 1            # 3. update

print("All transactions processed.")

This produces exactly what `for sale in sales:` would, with three lines of
bookkeeping you had to write yourself and could have got wrong. That is the
point of the comparison: **when you are walking a known collection, use a `for`
loop.** The `while` version is shown here so that the machinery a `for` loop
hides is visible at least once.

`index = index + 1` reads as "the new index is the old index plus one". The
shorthand `index += 1` means the same thing and is what you will normally write.

In [ ]:
# A sentinel-controlled example: the loop stops on a marker value.

responses = ["coffee", "tea", "done"]
index = 0

while responses[index] != "done":
    print(f"Customer selected: {responses[index]}")
    index += 1

print("No more selections.")

Here the repetition count is genuinely unknown to the code: it depends on where
`"done"` appears. A **sentinel** is a value that means "stop", and this is the
situation a `while` loop is actually for.

Notice the fragility, though — if `"done"` were missing from the list, the
condition would still be true when `index` reached 3 and the program would crash
with `IndexError`. Real sentinel loops guard against that, for example with
`while index < len(responses) and responses[index] != "done":`.

In [ ]:
# The danger: if the state never changes, the loop never ends.

count = 0

while count < 3:
    print(count)
    count += 1        # remove this line and the loop runs forever

print("Done.")

**Prompt.** Why is `count += 1` essential? In the transaction example, what plays
the role of the "progress" variable?

Without the update, `count` stays 0, `count < 3` stays true, and the loop never
terminates — an **infinite loop**. In the transaction example, `index` is the
progress variable: it advances each pass until it reaches `len(sales)` and the
condition fails.

If you do write an infinite loop in Jupyter — and you will — the notebook shows
`[*]` beside the running cell and never finishes. Stop it with the ■ button in
the toolbar, or **Kernel → Interrupt Kernel**. This is a normal event, not a
disaster: interrupt, fix the update, and run it again.

Rule of thumb: **`for` when you know the collection, `while` when you know only
the stopping condition.**

---

# Four loop patterns

Nearly every analysis you write this term is one of four patterns, or a
combination of them. Learn them by name — naming a pattern is what lets you
recognize a problem you have already solved.

| Pattern | Question it answers | Accumulator starts at |
|---|---|---|
| **Counting** | How many match? | `0` |
| **Accumulating** | What is the total? | `0.0` |
| **Filtering** | Which ones match? | `[]` |
| **Maximum** | Which is the largest? | the first element |

Every one has the same shape: **initialize before the loop, update inside the
loop, use the result after the loop.**

## 9. Pattern 1 — counting

1. Start a counter at zero.
2. Repeat over the collection.
3. Increment when the observation matches the condition.

In [ ]:
# Count coffee purchases.

coffee_count = 0

for sale in sales:
    if sale["item"] == "coffee":
        coffee_count += 1

print(f"Coffee purchases: {coffee_count}")

Watch the indentation, because it carries the meaning:

- `coffee_count = 0` is **outside** the loop. Inside, it would reset to 0 on
  every pass and the answer would always be 0 or 1.
- `coffee_count += 1` is **inside the `if`**, so it runs only on matches. Move it
  left by one level and you would count every transaction.

Note `==` (comparison) rather than `=` (assignment). Using `=` in an `if`
condition is a `SyntaxError` in Python — a small mercy that other languages do
not offer.

In [ ]:
# Count all drink purchases.

drink_count = 0

for sale in sales:
    if sale["category"] == "drink":
        drink_count += 1

print(f"Drink purchases: {drink_count}")

Same pattern, different field and different question. Only the condition
changed. That is the value of recognizing patterns: once you have written this
loop, "how many X" is never a new problem, only a new condition.

## 10. Pattern 2 — accumulating a total

Counting adds `1` per match. Accumulating adds a **value** per record. The
initial value is `0.0` rather than `0` because we are summing money.

In [ ]:
# Accumulate total revenue.

total_revenue = 0.0

for sale in sales:
    total_revenue += sale["price"]

print(f"Total revenue: ${total_revenue:.2f}")

There is no `if` here — every transaction contributes. `total_revenue +=
sale["price"]` is shorthand for `total_revenue = total_revenue + sale["price"]`.

Accumulating floats deserves one warning. Run `print(0.1 + 0.2)` in any cell and
Python answers `0.30000000000000004` — not a bug, but the consequence of storing
decimal fractions in binary. Prices like `3.50` happen to add up cleanly here,
but many do not, and the tiny errors accumulate across a long loop.

Two habits follow: format money with `:.2f` when you display it, and never test
two floats for exact equality with `==`.

In [ ]:
# Accumulate total revenue only from food.

food_revenue = 0.0

for sale in sales:
    
    if sale["category"] == "food":
        food_revenue += sale["price"]

print(f"Food revenue: ${food_revenue:.2f}")

Adding an `if` combines the accumulating pattern with a condition — this is
**conditional accumulation**, and it is the ancestor of pandas' `df[df.category
== "food"].price.sum()`. Same three ideas: select rows, take a column, add it up.

## 11. Pattern 3 — filtering into a new collection

Filtering builds a **new list** containing only the records that meet a
condition.

One rule matters enough to state on its own: **never remove items from a list
while you are iterating over it.** The loop tracks its position by index, and
removing an item shifts everything after it down one — so the loop skips
records, silently and without any error message. Build a new list instead.

In [ ]:
# Build a new list containing morning sales only.

morning_sales = []

for sale in sales:
    if sale["hour"] < 10:
        morning_sales.append(sale)

print(f"Morning transactions: {len(morning_sales)}")

for sale in morning_sales:
    print(sale)

The result is a list of the *same dictionaries* — not copies of them. `sales` is
untouched, so you can go on to ask other questions of the full dataset. But
because the dictionaries themselves are shared, editing one through
`morning_sales` would also change it in `sales`. Here we only read, so it does
not matter; it is worth knowing before you start modifying records.

Note also that `len(morning_sales)` answers "how many morning sales?" without a
separate counting loop. Filtering subsumes counting when you want the records
anyway.

In [ ]:
# Filter into a new list: food purchases made by students.

student_food_sales = []

for sale in sales:
    if sale["customer_type"] == "student" and sale["category"] == "food":
        student_food_sales.append(sale)

for sale in student_food_sales:
    print(f'{sale["day"]}: {sale["item"]} (${sale["price"]:.2f})')

`and` requires **both** conditions to hold. `or` requires at least one. `not`
inverts a condition. These are the Boolean operators from Session 2, doing real
work: a filter condition is just a Boolean expression evaluated once per record.

In [ ]:
# Equivalent list-comprehension form.
# Reach for this only once the explicit loop above is second nature.

morning_sales_compact = [sale for sale in sales if sale["hour"] < 10]

print(len(morning_sales_compact))
print(morning_sales_compact == morning_sales)   # same result

The comprehension packs the three-line pattern into one:

```text
[ sale            for sale in sales      if sale["hour"] < 10 ]
   ^ what to keep    ^ what to loop over    ^ which ones qualify
```

It is not faster to understand, only shorter to write, and a comprehension with
a complicated condition is worse than the loop it replaces. Use it when it fits
on one readable line.

## 12. Pattern 4 — finding a maximum

The maximum pattern keeps a "best so far" variable and replaces it whenever
something better turns up. The subtlety is what to initialize it with.

In [ ]:
# Find the most expensive single transaction.

most_expensive_sale = sales[0]

for sale in sales:
    if sale["price"] > most_expensive_sale["price"]:
        most_expensive_sale = sale

print("Most expensive transaction:")
print(most_expensive_sale)

In [ ]:
# Find the earliest transaction.

earliest_sale = sales[0]

for sale in sales:
    if sale["hour"] < earliest_sale["hour"]:
        earliest_sale = sale

print("Earliest transaction:")
print(earliest_sale)

A minimum is the same pattern with the comparison reversed. Nothing else
changes.

**Prompt.** Why initialize `most_expensive_sale` with `sales[0]` rather than the
number `0`?

Because the variable holds a **transaction**, not a price, and the comparison
`sale["price"] > most_expensive_sale["price"]` requires it to be a dictionary
from the very first pass. Starting at `0` would raise `TypeError: 'int' object is
not subscriptable`. The rule generalizes: **initialize the "best so far" with the
first element, not with a made-up value.**

If you only wanted the largest *price*, you could start at `0.0` and compare
numbers — but even then, starting with `prices[0]` is safer, because a made-up
starting value is wrong the moment your data can be negative (temperatures,
account balances, profit).

Both loops above compare against the first element again on the first pass,
which is harmless. And both keep the **first** record in a tie, since `>` is
strict — a tie-breaking rule you should be conscious of, because it decides what
your program reports when two records are equally good.

In [ ]:
### Try it: find the transaction with the LOWEST price.
### Then check your answer by eye against the dataset.

### Enter your code here ###

---

# Dictionary counting: the central Session 3 idiom

## 13. Build category counts carefully

The four patterns so far answer one question at a time: how many coffees, what
is total revenue. But the interesting question is usually "how many of *each*?"
— and that needs a dictionary whose keys are discovered from the data as the
loop runs.

The idiom has three steps inside the loop:

1. Read the value that should be the key.
2. If that key is not in the dictionary yet, create it with a starting value.
3. Increment it.

Step 2 is what beginners forget. Without it, `category_counts[category] += 1`
raises `KeyError` the first time each new category appears, because `+= 1` has
to read the old value before it can write a new one — and there is no old value.

In [ ]:
# Count how many transactions belong to each category.

category_counts = {}

for sale in sales:
    category = sale["category"]

    if category not in category_counts:
        category_counts[category] = 0

    category_counts[category] += 1

print(category_counts)

Trace the first three passes to see the dictionary being built:

| Pass | `category` | Key exists? | Dictionary after the pass |
|---|---|---|---|
| 1 | `drink` | no → create at 0 | `{'drink': 1}` |
| 2 | `drink` | yes | `{'drink': 2}` |
| 3 | `food` | no → create at 0 | `{'drink': 2, 'food': 1}` |

The dictionary starts **empty**: you do not need to know the categories in
advance, and you never have to edit a hard-coded list when the data gains a new
one. That is the whole point — the keys come from the data.

In [ ]:
# Report category counts clearly.

for category, count in category_counts.items():
    print(f"{category.title()}: {count} transactions")

Separating **computing** from **reporting** is a habit worth forming now. The
first loop produces a value; the second turns it into text for a human. Keeping
them apart means you can change the wording without touching the arithmetic, and
it is what makes each piece easy to test later.

## 14. Count item popularity, then find the winner

In [ ]:
# Count each item sold.

item_counts = {}

for sale in sales:
    item = sale["item"]

    if item not in item_counts:
        item_counts[item] = 0

    item_counts[item] += 1

print(item_counts)

In [ ]:
# Find the most frequent item by scanning the counts dictionary.

most_frequent_item = None
largest_count = 0

for item, count in item_counts.items():
    if count > largest_count:
        most_frequent_item = item
        largest_count = count

print(f"Most frequent item: {most_frequent_item}")
print(f"Number sold: {largest_count}")

This is the maximum pattern from Section 12, applied to a dictionary instead of
a list — and it is the standard way to answer "which key has the largest value?"

Two details deserve attention.

`most_frequent_item = None` uses `None` as an explicit "no answer yet". If
`item_counts` were empty, the loop would never run and the program would report
`None` — an honest answer — rather than crash or invent a value.

Starting `largest_count` at `0` is safe **here** because counts are never
negative and a count of at least 1 always beats it. That reasoning does not
transfer to prices, temperatures, or scores. When in doubt, initialize from the
first element.

The two variables must be updated **together**, inside the same `if`. Update one
without the other and the program will confidently report a wrong pairing.

## 15. The `.get()` shorthand

`dict.get(key, default)` returns the value for `key`, or `default` if the key is
absent. That collapses the "check, then create, then increment" idiom into one
line.

In [ ]:
# Same item-counting task using dict.get().

item_counts_get = {}

for sale in sales:
    item = sale["item"]
    item_counts_get[item] = item_counts_get.get(item, 0) + 1

print(item_counts_get)
print(item_counts_get == item_counts)   # identical result

Read the assignment right to left: *look up the current count, using 0 if the
item has never been seen; add 1; store it back under that key.*

The `if key not in ...` version and this one produce exactly the same
dictionary. Write whichever you can read at a glance — but understand the long
form first, because when a `KeyError` appears in your own code, the long form is
the shape of the fix.

The same shorthand works for accumulating, not just counting:
`revenue[cat] = revenue.get(cat, 0.0) + sale["price"]`.

## 16. `collections.Counter`: the same idea, packaged

`Counter` is a dictionary subclass built for exactly this job: it stores the
counted elements as keys and their counts as values. It is worth meeting *after*
the explicit idiom, not before — otherwise it is a black box that produces the
right answer for reasons you cannot explain.

Reference: [`collections`](https://docs.python.org/3/library/collections.html).

In [ ]:
from collections import Counter

items_sold = [sale["item"] for sale in sales]

item_counter = Counter(items_sold)

print(item_counter)
print(item_counter.most_common())
print(item_counter.most_common(1))

`from collections import Counter` imports one name from a module in Python's
standard library — nothing to install.

`Counter(items_sold)` does in one call what Section 14 did in six lines: walk
the list, and count. Because it *is* a dictionary, `item_counter["coffee"]` works
as you would expect — with one convenience: a missing key returns `0` instead of
raising `KeyError`.

`.most_common()` returns a list of `(item, count)` **tuples** sorted from most to
least frequent, and `.most_common(1)` returns just the top one — as a list of one
tuple, so the item itself is `item_counter.most_common(1)[0][0]`. That expression
is the argument for unpacking: `(top_item, top_count), = item_counter.most_common(1)`
says the same thing far more clearly.

In [ ]:
# Count transaction categories with Counter, straight from the records.

category_counter = Counter(sale["category"] for sale in sales)

for category, count in category_counter.items():
    print(f"{category.title()}: {count}")

The argument here is a **generator expression** — a comprehension without the
square brackets. It produces categories one at a time as `Counter` consumes
them, instead of building a whole intermediate list first. For twelve records
the difference is irrelevant; for twelve million it is the difference between
running and running out of memory.

## 17. List scan versus dictionary lookup

Both a list and a dictionary can answer "does this exist?" — but they are
answering **different questions**, and confusing the two produces code that is
right by accident.

In [ ]:
# A list scan: useful when the collection is a sequence of observed values.

if "coffee" in items_sold:
    print("At least one coffee was sold.")

In [ ]:
# A dictionary lookup: useful when we want the value associated with a key.

if "coffee" in menu_prices:
    print(f'Coffee price: ${menu_prices["coffee"]:.2f}')

In [ ]:
# A practical contrast:
# "coffee" in items_sold  answers: Did coffee occur in any transaction?
# "coffee" in menu_prices answers: Is coffee a defined menu item?

print("coffee" in items_sold)
print("coffee" in menu_prices)

An item can be on the menu and never sell; an item could be sold and be missing
from the menu — which is a data-quality problem, and exactly what question 10 of
the activity below checks for.

Note that `in` on a dictionary tests the **keys**, never the values.
`3.50 in menu_prices` is `False` even though it is a price. To search values you
need `in menu_prices.values()` — which is a scan, and a hint that the dictionary
may be oriented the wrong way round for the question you are asking.

---

# 18. In-class activity: coffee-cart mini-analysis

Work in pairs. Answer these using **core Python only** — no pandas, no
visualization library. Everything you need is above.

1. How many transactions occurred?
2. What distinct items and categories were sold?
3. How many purchases occurred for each item?
4. Which item was the most frequent?
5. What was total revenue?
6. Which day had the most transactions?
7. How many purchases came from students, faculty, and staff?
8. Which category produced more revenue: food or drinks?
9. What percentage of transactions happened before 10:00?
10. Is every item in the transaction data present in the menu-price dictionary?

**Try each one in the empty cell below before reading the worked solution.**
Getting stuck and then reading the answer teaches more than reading first.

For each question, ask yourself which pattern applies: counting, accumulating,
filtering, or maximum — and whether the answer is one number or one per key.

### Your Work

Complete these cells...

In [ ]:
# 1. Total number of transactions — no loop needed at all.



In [ ]:
# 2. Distinct items and categories — the set pattern, built with .add()



**Hint**

`set()` creates an **empty set**. Note that `{}` does *not* — it creates an empty
dictionary, which is a genuine trap. `.add()` is the set's equivalent of a
list's `.append()`, and adding a value that is already present simply does
nothing, which is the whole point.

In [ ]:
# 3. Counts by item — the counting pattern with .get()



In [ ]:
# 4. Most frequent item — the maximum pattern over a dictionary



In [ ]:
# 5. Total revenue — the accumulator pattern



In [ ]:
# 6. Transaction counts by day, then the busiest day —
#    counting, followed by maximum over the resulting dictionary.



**Hint**

Two loops, two patterns, in sequence: the first **counts into a dictionary**, the
second **finds the maximum** of that dictionary. Almost every "top N" question
you will ever be asked has this two-stage shape.

There is a tie here — check the counts and see. Because `>` is strict, the
program reports whichever tied day it met **first**, which is insertion order.
When ties are possible, decide deliberately what your program should report; a
tie-breaking rule that nobody chose is a bug waiting to be noticed.

In [ ]:
# 7. Counts by customer type — the same counting pattern, a third field.



In [ ]:
# 8. Revenue by category — accumulating INTO a dictionary
#    (counting adds 1; this adds the price)



This is the most important cell in the activity. It is **group-by-and-sum**,
written out by hand: scan the records, use one field as the key, and accumulate
another field into that key's running total.

In Session 8 you will write `df.groupby("category").price.sum()` and get the same
answer in one line. Having written the loop, you will know precisely what that
line does — and, more usefully, what it does with a category that appears only
once, or a missing price.

Note the starting value is `0.0`, not `0`: we are accumulating money, not
counting.

In [ ]:
# 9. Percentage before 10:00 — counting with a condition, then arithmetic.



`/` is true division and always produces a float, so `7 / 12` is
`0.5833...` rather than `0`. (`//` is floor division, which discards the
fraction — a common cause of percentages that come out as 0.)

Guard the division if the dataset could be empty: `len(sales)` of zero would
raise `ZeroDivisionError`.

In [ ]:
# 10. Data-quality check: does every sold item have a menu price?
#     Membership testing, plus filtering into a list.



This is the check that catches a whole class of real problems — a typo in one
record, a new product that never made it into the price table, an inconsistent
category name. Collecting into a **list** and reporting as a **set** is
deliberate: the list preserves how many times the problem occurred, and the set
answers *which distinct items* are affected.

**Extension.** Modify one record's `item` to `"smoothie"` and re-run this cell,
then re-run the counting cells. Notice that the counts still "work" — they
silently include a product that does not exist. Nothing crashes. That is why an
explicit data-quality check is worth writing.

---

## 19. Why this dataset, and what comes next

Everything in this notebook maps onto one coffee-cart question and one Python
structure:

| Session 3 concept | Coffee-cart question | Python structure or pattern |
|---|---|---|
| List | What is the ordered sequence of transactions? | `sales` |
| Dictionary | What fields describe one transaction? | `sale["item"]`, `sale["price"]` |
| Tuple | How can a fixed summary record be represented? | `("coffee", 4, 14.00)` |
| Set | What distinct items were sold? | `set(items_sold)` |
| `for` loop | Process every transaction | `for sale in sales:` |
| `while` loop | Process until the data stream ends | `while index < len(sales):` |
| Counting | How many of each item were sold? | `item_counts[item] += 1` |
| Accumulating | What is total revenue? | `total_revenue += sale["price"]` |
| Filtering | Which sales occurred before 10:00? | `if sale["hour"] < 10:` |
| Maximum | What is the most frequent item? | compare counts |
| Membership | Is an item on the menu? | `item in menu_prices` |
| `Counter` | What are the most common items? | `Counter(items_sold)` |

The same dataset becomes a CSV file in Session 5 and a pandas `DataFrame` in
Session 8. The questions will not change; only the amount of code will. That is
what makes this session worth the effort: you learn what a group-by *is* — scan
records, track keys, update values, find a maximum — before a library hides it
behind `value_counts()` or `groupby()`.

A library you cannot explain is a library you cannot debug.

---

## Summary

- A **list** is ordered, changeable, and allows duplicates. Indices start at 0;
  slices exclude the end. Build new lists rather than editing while iterating.
- A **tuple** is a fixed record. Immutability is a statement of intent, and
  unpacking (`a, b = pair`) is everywhere in Python.
- A **set** holds distinct values with no meaningful order. Use it for "which
  distinct values?" and "is this present?".
- A **dictionary** maps unique keys to values. `d[k]` raises `KeyError` on a
  missing key; `d.get(k, default)` does not.
- **`for`** when you know the collection; **`while`** when you know only the
  stopping condition — and always update the state a `while` condition depends
  on.
- The four patterns — **counting**, **accumulating**, **filtering**, **maximum**
  — all follow *initialize before, update inside, use after*.
- **Counting into a dictionary** discovers its keys from the data. The long form
  (`if key not in d`), the `.get()` shorthand, and `Counter` are three spellings
  of one idea.
- Finding "which key has the largest value" is the maximum pattern applied to a
  dictionary — the second half of every "top N" question.

### Before you close this notebook

1. **Kernel → Restart Kernel and Run All Cells…** and confirm it completes
   without errors. That is what makes the saved output trustworthy.
2. Save with `Ctrl+S` (`Cmd+S` on macOS).
3. Check the file name: it should be `Session_3_Loops_and_Data_Structures.ipynb`,
   with no `_orig`. A file that still has `_orig` in the name will be
   **overwritten** by the next `ifi8410-update`.

Homework for this session is **HW02** — see
`Assignments/HW02/instructions/homework02_instructions.md`. It applies these same
patterns to a different dataset, so if any section above is still unclear, go
back to it now rather than during the assignment.